# NLP with RNNs — Companion Notebook

**Course:** Sequences & Natural Language Processing with RNNs
#### (adapted from from Professsor Kyle Whynott's CPSC 483 course material and textbook:Hands-on Machine Learning by Geron, chapter 16)


In this notebook we'll overview:
1. **Char-RNN** — generating Shakespeare one character at a time, with temperature sampling.
2. **Stateful RNNs** — keeping hidden state across batches (why and how).
3. **Sentiment analysis (IMDb)** — word-level RNN classifier, padding, and masking.
4. **Pretrained embeddings** — plugging in Universal Sentence Encoder from TF Hub.
5. **Encoder–Decoder NMT** — translating English→Spanish with teacher forcing.
6. **Beam search** — a better decoder than greedy.

## 0. Imports and Initial Setup

In [1]:
import os, random, time, re, io, zipfile, urllib.request, glob
import numpy as np
import tensorflow as tf
from tensorflow import keras

# Random Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Run-length knobs (keep small for demo; bump for better quality)
EPOCHS_CHAR = 1
EPOCHS_IMDB = 3
EPOCHS_NMT  = 3

print("TF version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


TF version: 2.20.0
GPU available: False


# 1. Char-RNN — Generating Shakespeare

### Self-supervision

Training data is **the text itself**: input is a window of N characters, target is the same window *shifted by one*. That means every unlabeled corpus is training data — this is the trick that powers modern language models.

$$\text{Given: } c_1, c_2, \ldots, c_N \quad\Longrightarrow\quad \text{Predict: } c_2, c_3, \ldots, c_{N+1}$$

You might see this framed as "predict the next token given the prefix"; the windowed version simply trains on *every* prefix in parallel.

In [2]:
# Download Shakespeare — the tiny Shakespeare corpus, ~1.1M characters
# Géron's mirror from the textbook

# Note: This also shows how to fetch a text file from a live url

SHAKESPEARE_URL = "https://homl.info/shakespeare"
path = keras.utils.get_file("shakespeare.txt", SHAKESPEARE_URL)
with open(path) as f:
    shakespeare_text = f.read()

print(f"{len(shakespeare_text):,} characters")
print("--- sample ---")
print(shakespeare_text[:250])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1,115,394 characters
--- sample ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



## 1.2 Tokenizing at the Character Level (pre-process input data)

`TextVectorization` is the standard Keras preprocessor. This is similar to the `TFID_Vectorizer`
  we saw earlier with Scikit-Learn.

Crucial Settings to Note Here:

- `split="character"` — each character is a token.
- `standardize="lower"` — lowercase everything (halves the vocabulary).

`TextVectorization` makes reservations for:
- **ID 0** for *padding* (more on this later in sentiment).
- **ID 1** for *unknown* tokens (rare for char-level; common for word-level).

So when you see `vocab_size - 2` below, that's the count of real characters in the text. And the targets we train on are IDs shifted down by 2 so the model's softmax doesn't have to waste probability mass on pad/unknown.

Note:In NLP, we usually call a piece of input data "token", instead of input feature. Although Token = one unit (piece) of that feature after text is split

In [3]:
#Create tokenizer
text_vec_layer = keras.layers.TextVectorization(
    split="character", standardize="lower"
)
#Learn the vocabulary + finds all unique characters +  asssigns each one an ID
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0].numpy()

# Drop pad/unknown reservations
encoded = encoded - 2
#Original IDs: 0: reserved for padding, 1: reserved for OOV (out of vocabulary), after -2, real chars start at 0
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded)

print(f"vocabulary size (real chars): {n_tokens}")
print(f"dataset length: {dataset_size:,} chars")
print("first 25 encoded:", encoded[:25])
print("decoded back:", repr("".join(text_vec_layer.get_vocabulary()[i + 2] for i in encoded[:25]))) # repr(...) showns special characters like \n, \t

vocabulary size (real chars): 39
dataset length: 1,115,394 chars
first 25 encoded: [19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
  0]
decoded back: 'first citizen:\nbefore we '


## 1.3 Windowing — Turning a Stream into Training Examples

A Language Model sees **fixed-length windows** of the text.

For each window of length `L`, the input is characters `0..L-1` and the target is characters `1..L` (shift by one).

### Why Windows Instead of One Long Sequence?

There are a few reasons for this, batch processing is far easier and more managable for the CPU and GPU to handle as Backpropagation Through Time can become very computationally expensive (Remeber there is 1.1 million characters). This also makes gradients easier to compute,

Something to note, is that the model **does not see** cross-window dependencies during training, that's what **stateful RNNs** are for.

### Shuffle vs. sequential

Windows are **shuffled** for stateless training. Shuffling decorrelates batches and gives better gradients, at the cost of losing long-range order, which doesn't matter here for character sequence processing.

In [4]:
def make_windows(sequence, length, shuffle=False, seed=None, batch_size=32):
    # tf.data is the canonical way to build stream->window->batch pipelines.
    # from_tensor_slices turns the array into a dataset of individual tokens.
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    # window(L+1) groups L+1 consecutive tokens; drop_remainder keeps shapes uniform.
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    # window size is L+1 because we want to split it into (input, target) pairs of length L; shift=L means no overlap between windows

    # Each window is itself a nested dataset; flatten it to tensors of length L+1.
    ds = ds.flat_map(lambda w: w.batch(length + 1))

    if shuffle:
        ds = ds.shuffle(buffer_size=100_000, seed=42)
    # Split each window into (input, target) via the shift-by-one rule.
    ds = ds.map(lambda w: (w[:-1], w[1:]))
    # The function lambda w: (w[:-1], w[1:]) takes one window w and splits it into a tuple of two tensors:
    #w[:-1] (The Input / X)
    # w[1:] (The Target / Y)
    return ds.batch(batch_size).prefetch(1)

LENGTH = 100
train_ds = make_windows(encoded[:1_000_000], LENGTH, shuffle=True, seed=42)
valid_ds = make_windows(encoded[1_000_000:1_060_000], LENGTH)
test_ds  = make_windows(encoded[1_060_000:], LENGTH)

# Peek at one batch so you see the shapes
for xb, yb in train_ds.take(1):
    print("input batch shape :", xb.shape)   # (32, 100)
    print("target batch shape:", yb.shape)   # (32, 100)
    print("example x[0][:20] :", xb[0, :20].numpy())
    print("example y[0][:20] :", yb[0, :20].numpy())
    print("↑ y is x shifted left by 1")

input batch shape : (32, 100)
target batch shape: (32, 100)
example x[0][:20] : [ 3  9 17 10 16  6  3 14  0  5  0 13  9  9  4  2 13  8  4 11]
example y[0][:20] : [ 9 17 10 16  6  3 14  0  5  0 13  9  9  4  2 13  8  4 11 11]
↑ y is x shifted left by 1


## 1.4 The model

Three layers, that's it:

1. **`Embedding(n_tokens, 16)`** — maps each integer character ID to a 16-d learned vector. 16 is small on purpose: there are only ~40 characters, so we don't need much capacity here. (num. token: number of unique characters in the dataset)
2. **`GRU(128, return_sequences=True)`** — the recurrent core. `return_sequences=True` means *emit one output per time step*, not just the final one. We need the per-step outputs because we want to predict the next character at *every* position.
3. **`Dense(n_tokens, activation="softmax")`** — one classifier per time step. The softmax gives us a probability distribution over the next character.

### Why GRU, not LSTM or SimpleRNN?

- Remeber, **Simple RNNs** __die__ on any non-trivial sequence due to the vanishing/exploding gradient problem.
- **LSTM** is a great choice, but is more computationally expensive than GRU.
- **GRU** trains faster and uses fewer parameters. It is essentially as good as LSTM in practice.

In [5]:
def build_char_model(vocab, embed_dim=16, hidden=128):
    return keras.Sequential([
        keras.layers.Embedding(input_dim=vocab, output_dim=embed_dim),
        keras.layers.GRU(hidden, return_sequences=True),
        keras.layers.Dense(vocab, activation="softmax"),
    ])

char_model = build_char_model(n_tokens)
char_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"],
)
char_model.summary()

# you can build each layer separately
'''

inputs = keras.Input(shape=(None,), dtype="int32") # just a placeholder for future data”
x = keras.layers.Embedding(input_dim=n_tokens, output_dim=16)(inputs) #calls the layer on the input, similar to f = Embedding(...) then x = f(inputs)
x = keras.layers.GRU(128, return_sequences=True)(x)
outputs = keras.layers.Dense(n_tokens, activation="softmax")(x)

char_model = keras.Model(inputs=inputs, outputs=outputs)

char_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"],
)

char_model.summary()
'''

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

'\n\ninputs = keras.Input(shape=(None,), dtype="int32") # just a placeholder for future data”\nx = keras.layers.Embedding(input_dim=n_tokens, output_dim=16)(inputs) #calls the layer on the input, similar to f = Embedding(...) then x = f(inputs)\nx = keras.layers.GRU(128, return_sequences=True)(x)\noutputs = keras.layers.Dense(n_tokens, activation="softmax")(x)\n\nchar_model = keras.Model(inputs=inputs, outputs=outputs)\n\nchar_model.compile(\n    loss="sparse_categorical_crossentropy",\n    optimizer="nadam",\n    metrics=["accuracy"],\n)\n\nchar_model.summary()\n'

## 1.5 Train

We set `EPOCHS_CHAR=1` below so that this cell finishes quickly; for usable generations you want **5-10** epochs.

**What to Take Note of:** loss should drop below ~2.0 within a few epochs. That's well above random (log(39) ≈ 3.66) but well below what a good model achieves (~1.0). Don't expect Shakespeare from one epoch — expect *words that look like English*.

In [6]:
history = char_model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=1,
)

    313/Unknown 159s 225ms/step - accuracy: 0.2051 - loss: 2.9413

c:\Users\Troy\Downloads\cpsc483_spring_2026\lab-venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


313/313 ━━━━━━━━━━━━━━━━━━━━ 168s 252ms/step - accuracy: 0.2650 - loss: 2.6119 - val_accuracy: 0.3346 - val_loss: 2.2545


### Note: Some important block diagram of the model
#### Here are some key code we have in the model
![image.png](attachment:image.png)

#### The model architecture is as follows:

![image-2.png](attachment:image-2.png)

#### We will be using the ouput probablity to generate the prediction of next characters

## 1.6 Generating text — softmax, greedy, sampling, temperature

A trained language model gives you `P(next_char | prefix)`. Generation is an iterative loop:

```
for step in range(N):
    probs = model(prefix)[:, -1]          # distribution over next char
    next_char = pick_one(probs)           # ← decoding strategy
    prefix = prefix + next_char
```

The `pick_one` step is where it gets interesting.

### Greedy Decoding

Always pick the highest-probability char. **Problem:** the model gets stuck in loops ("the the the the..."). Greedy is cheap but boring.

### Random Sampling

Draw from the full distribution. **Problem:** rare-but-plausible characters sometimes chain into nonsense.

### Temperature Sampling

Sharpen or flatten the distribution before sampling:

$$p_i \leftarrow \frac{\exp(\log p_i / T)}{\sum_j \exp(\log p_j / T)}$$

- $T \to 0$: approaches greedy (confident but repetitive).
- $T = 1$: true distribution.
- $T > 1$: flatter, more creative, more mistakes.

This is the **single most important inference knob** in language modeling. ChatGPT's `temperature` parameter is the same mechanism.

Note: during training, the window size is 100. However, the prefix length can be any number during testing

In [ ]:
# Wrap the char model so we can feed raw strings through a single call.
# We use the **functional API** so the TextVectorization layer (which takes
# strings) and the char_model (which takes integer IDs) compose into one Model.
#
# Keras 3 note: `.predict()` currently trips on string-dtype inputs via its
# data adapter (the `Invalid dtype: strXXX` error). The fix is to invoke the
# model DIRECTLY — `shakespeare_generator(tf.constant([...]), training=False)`
# — instead of going through `.predict()`. Semantically identical; just skips
# the batching/logging machinery that doesn't know how to tree_map over strings.

text_in = keras.layers.Input(shape=(), dtype=tf.string) #placeholder for future string data
ids = text_vec_layer(text_in)
# Shift IDs to match training (we trained on ids - 2). Clamp at 0 so the
# UNK id (originally 1, now -1) doesn't blow up the Embedding.
ids = keras.layers.Lambda(lambda x: tf.maximum(x - 2, 0),
                          output_shape=lambda s: s)(ids)
out = char_model(ids)
shakespeare_generator = keras.Model(text_in, out)

def next_char(prefix: str, temperature: float = 1.0) -> str:
    # DIRECT call (not .predict()) — avoids the Keras 3 string-input issue.
    probs = shakespeare_generator(tf.constant([prefix]), training=False).numpy()[0, -1]
    # tf.random.categorical expects logits; convert from the softmax probs.
    logits = np.log(np.clip(probs, 1e-12, 1.0)) / temperature #log() converts probs → logits, apply temperature
    next_id = int(tf.random.categorical(
        logits[None, :].astype("float32"), num_samples=1)[0, 0].numpy())
    return text_vec_layer.get_vocabulary()[next_id + 2] # +2 to undo the earlier -2 shift

def extend(prefix: str, n_chars: int = 50, temperature: float = 1.0) -> str:
    for _ in range(n_chars):
        prefix += next_char(prefix, temperature)
    return prefix

# Try several temperatures — low T ≈ greedy/repetitive, high T ≈ chaotic.
for T in [0.2, 0.8, 1.5]:
    tf.random.set_seed(42)
    print(f"T={T}:")
    print(extend("to be or not to b", 100, temperature=T))  #extend(...) function is repeatedly calling next_char() and appending characters


T=0.2:


### Expected Behavior

- **T=0.2** → text repeats and commits strongly to common characters (lots of spaces and "the"s).
- **T=0.8** → the sweet spot for most generation; English-like.
- **T=1.5** → nonsense words sneak in.

### Note on this demo:

With only `EPOCHS_CHAR=1`, the outputs will still be very rough. Rerun with `EPOCHS_CHAR=10` and the quality jumps dramatically — you'll start seeing pseudo-Shakespeare.

### Why this matters Beyond Character RNNs

The same decoding logic — greedy vs. sampling vs. temperature vs. beam search (coming later) — is used by every modern LM, including GPT-style Transformers. The mechanism we just implemented is not obsolete; only the underlying model is.

# 2. Stateful RNNs

## 2.1 The Problem with our Char-RNN

When we shuffled windows and started each batch with the hidden state reset to zero, the model could only ever see **100 characters of context**. It has no way to remember "the narrator is Hamlet" from ten windows ago.

A **stateful RNN** fixes this by keeping the hidden state **across windows (batches)**. You lose shuffling (batches must be consecutive chunks of the sequence), but you gain unbounded context in principle.

## 2.2 Note: Two things that Change

1. **Dataset**: windows must be **consecutive, non-overlapping**, and batch `b`'s windows must line up sequentially with batch `b-1`'s. No shuffling.
2. **Model**: `GRU(..., stateful=True, batch_input_shape=(B, L))`. Batch size is now baked in because the layer keeps a hidden state tensor of shape `(B, L)`.

### Resetting Between Epochs

You do NOT want state to leak from the end of epoch 1 to the start of epoch 2 when the text restarts; however, there is no way for the hidden state to know that. We mus reset state manually with a callback.

In [ ]:
# Build a stateful dataset over a small slice so the demo runs quickly.
# This mirrors Géron §16.1.3 almost line-for-line.

def stateful_dataset(sequence, length, batch_size=32):
    # chunk the stream into batch_size consecutive substreams
    chunks = np.array_split(sequence[: (len(sequence) // batch_size) * batch_size],
                            batch_size)
    # build windows from each chunk, zip them as parallel streams
    datasets = [
        tf.data.Dataset.from_tensor_slices(chunk)
        .window(length + 1, shift=1, drop_remainder=True)
        .flat_map(lambda w: w.batch(length + 1))
        .map(lambda w: (w[:-1], w[1:]))
        for chunk in chunks
    ]
    return tf.data.Dataset.zip(tuple(datasets)).map(
        lambda *pairs: (tf.stack([p[0] for p in pairs]), tf.stack([p[1] for p in pairs]))
    ).prefetch(1)

BATCH = 32
stateful_train = stateful_dataset(encoded[:200_000], length=LENGTH, batch_size=BATCH)

# Keras 3: pin the batch dim via an Input layer instead of batch_input_shape.
inp = keras.layers.Input(batch_shape=(BATCH, None), dtype="int64")
x = keras.layers.Embedding(input_dim=n_tokens, output_dim=16)(inp)
x = keras.layers.GRU(128, return_sequences=True, stateful=True)(x)
out = keras.layers.Dense(n_tokens, activation="softmax")(x)
stateful_model = keras.Model(inp, out)

class ResetStatesCallback(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        for layer in self.model.layers:
            if hasattr(layer, "reset_states"):
                layer.reset_states()

stateful_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"],
)

# One epoch as a proof-of-life — full stateful training is slow on CPU
stateful_model.fit(
    stateful_train,
    epochs=3,
    callbacks=[ResetStatesCallback()],
)


In [ ]:
import numpy as np
import tensorflow as tf

# Get vocabulary list (used to convert IDs back to characters)
vocab = text_vec_layer.get_vocabulary()

# =========================
# Reset hidden states
# =========================
def reset_states(model):
    # Stateful RNN keeps memory → must reset before generating a new sentence
    for layer in model.layers:
        if hasattr(layer, "reset_states"):
            layer.reset_states()

# =========================
# Sampling helper
# =========================
def sample_from_probs(probs, temperature=1.0):
    # Convert probabilities → logits and apply temperature
    logits = np.log(np.clip(probs, 1e-12, 1.0)) / temperature

    # Sample one index based on probability distribution
    return int(tf.random.categorical(logits[None, :], 1)[0, 0].numpy())

# =========================
# Generate text using stateful model
# =========================
def generate_stateful(prefix, n_chars=100, temperature=1.0):
    reset_states(stateful_model)  # clear old memory

    result = prefix

    # Convert input string → integer IDs
    ids = text_vec_layer(tf.constant([prefix]))

    # Match training preprocessing (we trained on ids - 2)
    ids = tf.maximum(ids - 2, 0)

    # IMPORTANT:
    # Model was trained with batch size = 32
    # So we must repeat the same input 32 times to match shape
    ids = tf.repeat(ids, repeats=BATCH, axis=0)

    # Feed full prefix once (initialize hidden state)
    probs = stateful_model(ids, training=False).numpy()[0, -1]

    for _ in range(n_chars):
        # Sample next character ID
        next_id = sample_from_probs(probs, temperature)

        # Convert ID → actual character
        next_char = vocab[next_id + 2]  # +2 reverses earlier -2 shift
        result += next_char

        # Convert new character → ID
        new_ids = text_vec_layer(tf.constant([next_char]))
        new_ids = tf.maximum(new_ids - 2, 0)

        # Again repeat to match batch size = 32
        new_ids = tf.repeat(new_ids, repeats=BATCH, axis=0)

        # Feed ONLY the new character (state carries context)
        probs = stateful_model(new_ids, training=False).numpy()[0, -1]

    return result

# =========================
# Test with different temperatures
# =========================
for T in [0.2, 0.8, 1.5]:
    tf.random.set_seed(42)
    print(f"T={T}:")
    print(generate_stateful("to be or not to b", 100, temperature=T))
    print()

## 2.3 When to use stateful RNNs

**To be frank, not often these days.**

- For very long documents, **Transformers with sliding-window attention or memory tokens** usually beat stateful RNNs.
- For **streaming inference (e.g., online speech recognition), stateful RNNs are still useful** — you literally *need* to keep state across chunks.
- For generation from a huge context, modern approaches (**long-context Transformers, retrieval-augmented generation**) have largely displaced this pattern.

The concept of "carrying state across batches" remains important in Transformer variants with KV-cache.

# 3. Returning to Sentiment Analysis on IMDb

## 3.1 The task

- **Input:** a movie review (variable length, ~100-1000 words).
- **Output:** a binary label — positive or negative.
- **Dataset:** IMDb reviews via TensorFlow Datasets. 25k train, 25k test, balanced.

This is a **multi-label** RNN task: read a whole sequence, produce a single label.

## 3.2 What's New from the Char-RNN (and the Logistic One-vs-Rest Classifier)

| | Char-RNN | Sentiment |
|--|--|--|
| Task | next-char | classification |
| Output | sequence | single prediction |
| Unit | character | word |
| Vocabulary | ~40 | ~10,000 |
| Sequence length | fixed (100) | **variable** |
| Padding | unnecessary | **required** |
| Masking | unnecessary | **required** |

**Padding and masking are the conceptual wrinkle here.** Different reviews have different lengths; **a batch needs tensors of uniform shape**, so we pad short reviews with 0s. But then the model *sees* the zeros and learns to treat them as signal — which is wrong. **Masking** tells the RNN "ignore timesteps where the input is 0."

In [ ]:
import tensorflow.datasets as tfds

# Load IMDb. We use as_supervised=True so each example is (text, label).
raw_train, raw_valid, raw_test = tfds.load(
    "imdb_reviews",
    split=["train[:90%]", "train[90%:]", "test"],
    as_supervised=True,
)

# Peek
for text, label in raw_train.take(2):
    print(f"[{label.numpy()}] {text.numpy()[:180]}...")

## 3.3 Vocabulary and Vectorization

We cap the vocabulary at 10,000 words. `TextVectorization` will assign IDs to the 10k most frequent words, map everything else to the **OOV** token (ID 1), and reserve ID 0 for padding.

`output_sequence_length=200` **truncates** long reviews to 200 tokens and **pads** short ones to 200.

**Something to Note:** We could skip padding here (tf.data handles ragged tensors), but the fixed length makes the demo simpler.

In [ ]:
VOCAB_SIZE = 10_000
SEQ_LEN    = 200

text_vec = keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQ_LEN,
)
# adapt() walks the training texts to build the vocabulary (frequency-ranked)
text_vec.adapt(raw_train.map(lambda text, label: text))

print("vocab size:", text_vec.vocabulary_size())
print("sample tokens:", text_vec.get_vocabulary()[:15])

## 4.4 Why masking matters — a quick demo

Without masking, the padded zeros flow through the RNN and shift the hidden state. Let's show this concretely with a toy sequence.

In [ ]:
# Build two models: one with mask_zero=True, one without.
def make_demo_model(mask_zero):
    return keras.Sequential([
        keras.layers.Embedding(input_dim=50, output_dim=4, mask_zero=mask_zero),
        keras.layers.GRU(4),
    ])

no_mask  = make_demo_model(False)
with_mask = make_demo_model(True)

# IMPORTANT: Keras lazily builds weights on first call. We must force both
# models to build BEFORE copying weights, otherwise `get_weights()` returns []
# and the two models end up with different random inits — which would make
# this demo "work" for the wrong reason.
_ = no_mask(np.zeros((1, 8), dtype="int64"))
_ = with_mask(np.zeros((1, 8), dtype="int64"))
with_mask.set_weights(no_mask.get_weights())

# Two inputs: identical real content, different amounts of padding on the right.
x1 = np.array([[3, 7, 2, 9, 0, 0, 0, 0]])               # 4 real tokens, 4 pad
x2 = np.array([[3, 7, 2, 9, 0, 0, 0, 0, 0, 0, 0, 0]])   # same content, more pad

print("No mask — states should DIFFER if pads leak in:")
print("  x1:", no_mask(x1).numpy().round(3))
print("  x2:", no_mask(x2).numpy().round(3))

print("\nWith mask — states should be IDENTICAL:")
print("  x1:", with_mask(x1).numpy().round(3))
print("  x2:", with_mask(x2).numpy().round(3))


You should see the no-mask outputs differ slightly between `x1` and `x2` — the padding was "processed" by the GRU and shifted its state. The masked version ignores padded positions entirely, so the outputs match.

**Rule of thumb:** *always* set `mask_zero=True` on embedding layers that see padded sequences. The only reason not to is if you're sure all sequences have the same length.

## 4.5 The classifier

Architecture:

```
text -> TextVectorization -> Embedding(mask_zero=True)
     -> GRU(64)  (many-to-one: we take the final state)
     -> Dense(1, sigmoid)
```

Binary classification → sigmoid + binary cross-entropy.

In [ ]:
BATCH = 32

def encode_batch(text, label):
    return text_vec(text), label

train_enc = raw_train.batch(BATCH).map(encode_batch).prefetch(1)
valid_enc = raw_valid.batch(BATCH).map(encode_batch).prefetch(1)
test_enc  = raw_test.batch(BATCH).map(encode_batch).prefetch(1)

sentiment_model = keras.Sequential([
    keras.layers.Embedding(VOCAB_SIZE, 64, mask_zero=True),
    keras.layers.GRU(64),
    keras.layers.Dense(1, activation="sigmoid"),
])
sentiment_model.compile(
    loss="binary_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"],
)

sentiment_model.fit(
    train_enc,
    validation_data=valid_enc,
    epochs=EPOCHS_IMDB,
)

## 4.6 Evaluate and try a new review

In [ ]:
loss, acc = sentiment_model.evaluate(test_enc, verbose=0)
print(f"test loss: {loss:.3f}  accuracy: {acc:.3f}")

demo_reviews = [
    "This film was a masterpiece — the acting, the direction, everything clicked.",
    "A total waste of two hours. I want my money back.",
    "It was... fine. Some good scenes, some bad, mostly forgettable.",
]
preds = sentiment_model.predict(text_vec(demo_reviews), verbose=0).flatten()
for r, p in zip(demo_reviews, preds):
    label = "POSITIVE" if p > 0.5 else "NEGATIVE"
    print(f"{label} ({p:.2f}) — {r[:80]}")

## 4.7 Things to think about

- **Accuracy ceiling:** A simple GRU on IMDb caps out around 85-88%. Transformer-based models (BERT fine-tuned) reach ~95%. The gap is mostly about long-range context and pretraining.
- **Bidirectional is better here:** for classification you don't need causal masking, so a bidirectional RNN (reading the review left-to-right *and* right-to-left) typically wins. Try wrapping the GRU in `keras.layers.Bidirectional(...)`.
- **Neutral reviews trip up binary classifiers** — see the "It was... fine" example. Real sentiment systems are rarely binary.

# 5. Pretrained Embeddings and Language Models

## 5.1 Why reuse?

Training good word embeddings from scratch takes a huge corpus. Research labs have already done this work — so use their embeddings. This is **transfer learning for NLP**, and it's a free lunch for almost any task.

### The pretrained embeddings provided

- **Word2Vec (2013)** — static word vectors, trained to predict neighbors (skip-gram) or fill blanks (CBOW). One vector per word, no context.
- **GloVe (2014)** — static word vectors trained on global co-occurrence statistics. Still one vector per word.
- **ELMo (2018)** — contextual embeddings from a biLSTM language model. Different vector for "bank" in "river bank" vs. "savings bank."
- **BERT (2018)** — contextual embeddings from a Transformer. The new default.
- **GPT / LLaMA / ...** — decoder-only Transformers, used as both feature extractors and generators.

For a plug-and-play sentence-level embedding that works reasonably well without fine-tuning, **Universal Sentence Encoder** from TF Hub is a good baseline.

## 5.2 Example use with a small classifier

We replace the Embedding + GRU stack with a single pretrained encoder that turns a whole sentence into a 512-d vector, then train a small MLP head. This is much less data-hungry than training from scratch.

In [ ]:
# Universal Sentence Encoder from TF Hub.
# Needs a network round-trip (~900 MB on first run). This cell is guarded so
# it's safe to skip when offline.
#
# Keras 3 change: the Sequential-with-input_shape=[] pattern from Géron's book
# is brittle on Keras 3. The functional API is more reliable for string inputs
# — define an Input(shape=(), dtype=tf.string) and wire the USE layer to it.
try:
    import tensorflow_hub as hub
    USE_URL = "https://tfhub.dev/google/universal-sentence-encoder/4"
    use_layer = hub.KerasLayer(USE_URL, trainable=False)

    text_in = keras.layers.Input(shape=(), dtype=tf.string)
    emb = use_layer(text_in)                    # sentence -> 512-d vector
    h = keras.layers.Dense(64, activation="relu")(emb)
    out = keras.layers.Dense(1, activation="sigmoid")(h)
    use_model = keras.Model(text_in, out)

    use_model.compile(
        loss="binary_crossentropy",
        optimizer="nadam",
        metrics=["accuracy"],
    )

    # Train on raw text directly — USE handles tokenization internally.
    use_train = raw_train.batch(BATCH).prefetch(1)
    use_valid = raw_valid.batch(BATCH).prefetch(1)

    use_model.fit(use_train, validation_data=use_valid, epochs=1)
except Exception as e:
    print("TF Hub / USE demo skipped:", e)


## 5.3 The core idea

You don't have to train embeddings from scratch. Nearly every NLP task can start from a pretrained model and fine-tune a small head. For modern applications, you'd use `transformers` from Hugging Face to load BERT or a similar model; USE is just a convenient one-line demo.

# 6. Encoder–Decoder Neural Machine Translation

## 6.1 Sequence-to-sequence

NMT is the task of producing a **sequence** from a **sequence** — specifically, producing a translation from a source sentence.

The encoder-decoder pattern:
1. **Encoder** reads the source sentence and compresses it to a fixed-size vector (the *context* or *thought vector*).
2. **Decoder** is a conditional language model: generates the target sentence token by token, starting from the context vector.

### Teacher Forcing

At training time, the decoder is trained to predict token `t+1` given **ground-truth** tokens `1..t` (but not its own previous predictions). This is called **teacher forcing**:

- **Pros:** training is stable and parallelizable across time steps
- **Cons**: at inference the decoder sees its own outputs, which can drift from the training distribution (*exposure bias*)

Modern models mitigate exposure bias with scheduled sampling or RL-based fine-tuning, but for a first pass teacher forcing is the standard approach.

## 6.2 A tiny English→Spanish dataset

We use the Anki sentence-pairs dataset (used in many TF tutorials). It's small enough to train in a few minutes and large enough to demonstrate the pipeline.

In [ ]:
# Download the Anki Spanish pairs (~120k sentence pairs)
URL = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path_to_zip = keras.utils.get_file("spa-eng.zip", origin=URL, extract=True)

# Keras 3 extracts into a sibling "<name>_extracted/" dir. Find spa.txt robustly.
search_root = os.path.dirname(path_to_zip)
matches = glob.glob(os.path.join(search_root, "**", "spa.txt"), recursive=True)
if not matches:
    raise FileNotFoundError(f"spa.txt not found under {search_root}")
path_to_file = matches[0]
print("using:", path_to_file)

pairs = []
with open(path_to_file, "r", encoding="utf-8") as f:
    for line in f:
        eng, spa = line.strip().split("\t")
        pairs.append((eng.lower(), spa.lower()))

print(f"loaded {len(pairs):,} pairs")
print("examples:")
for p in pairs[:3]: print(" ", p)

# Keep it small for demo speed
random.Random(SEED).shuffle(pairs)
pairs = pairs[:30_000]
eng_sentences, spa_sentences = zip(*pairs)

# Add start/end tokens to the Spanish side — these are what the decoder conditions on
spa_sentences = [f"startofseq {s} endofseq" for s in spa_sentences]


## 6.3 Build separate vocabularies for source and target

Separate `TextVectorization` layers for English and Spanish. Each has its own vocabulary, own padding, own OOV.

In [ ]:
VOCAB_SRC = 8000
VOCAB_TGT = 8000
MAX_LEN   = 30

src_vec = keras.layers.TextVectorization(max_tokens=VOCAB_SRC, output_sequence_length=MAX_LEN)
tgt_vec = keras.layers.TextVectorization(max_tokens=VOCAB_TGT, output_sequence_length=MAX_LEN + 1)

src_vec.adapt(list(eng_sentences))
tgt_vec.adapt(list(spa_sentences))

print("English vocab:", src_vec.vocabulary_size())
print("Spanish vocab:", tgt_vec.vocabulary_size())
# Sanity: start/end tokens should be in the vocab
tgt_vocab = tgt_vec.get_vocabulary()
print("startofseq id:", tgt_vocab.index("startofseq"))
print("endofseq id:  ", tgt_vocab.index("endofseq"))

## 6.4 Training data layout

For teacher forcing we need three tensors per example:

- `encoder_input`   = English sentence (length MAX_LEN)
- `decoder_input`   = Spanish sentence **shifted right by one** — starts with `startofseq`
- `decoder_target`  = Spanish sentence **as is** — ends with `endofseq`

The model is trained to output `decoder_target` given both encoder_input and decoder_input.

(Why not just feed the Spanish sentence to the decoder? Because then at time step `t`, the input would already include the answer. Teacher forcing shifts the input so position `t` sees tokens `0..t-1` and must predict token `t`.)

In [ ]:
# Build encoder/decoder arrays up-front with numpy — dataset is small
# enough (~30k pairs) to fit in memory, and a single vectorize pass is cleaner
# than a tf.data map with a numpy_function.
#
# Teacher forcing lives here: the decoder input is the target shifted right
# by one position (starts with `startofseq`), and the decoder target is the
# target shifted left by one (ends with `endofseq`). At training time the
# decoder always sees the ground-truth previous token, regardless of what
# the model would have predicted.

enc_input_arr = src_vec(np.array(eng_sentences)).numpy()
spa_full_arr  = tgt_vec(np.array(spa_sentences)).numpy()
dec_input_arr  = spa_full_arr[:, :-1]   # "startofseq ... <last-1>"
dec_target_arr = spa_full_arr[:, 1:]    # "<first+1> ... endofseq"

print("encoder_input shape:", enc_input_arr.shape)
print("decoder_input shape:", dec_input_arr.shape)
print("decoder_target shape:", dec_target_arr.shape)
print("\nFirst pair (tokenized):")
print("  eng:", eng_sentences[0])
print("  spa:", spa_sentences[0])
print("  enc_in [:10]:", enc_input_arr[0][:10])
print("  dec_in [:10]:", dec_input_arr[0][:10])
print("  dec_tgt[:10]:", dec_target_arr[0][:10])


## 6.5 The model

Two sequences in, one sequence out. Functional API makes this clean.

```
encoder_in ---Embedding---LSTM---> [state_h, state_c]
                                          |
                                          v
decoder_in ---Embedding---LSTM(initial_state=...)---Dense(softmax)---> decoder_out
```

The encoder's final state is the bottleneck — a single vector that must capture the entire source sentence. This is exactly the limitation attention solves next lecture.

In [ ]:
EMB = 128
UNITS = 256

# ---- Encoder ----
enc_inputs = keras.layers.Input(shape=(MAX_LEN,), dtype="int64", name="encoder_input")
x = keras.layers.Embedding(VOCAB_SRC, EMB, mask_zero=True)(enc_inputs)
# return_state=True gives (output, state_h, state_c) for LSTM.
_, state_h, state_c = keras.layers.LSTM(UNITS, return_state=True)(x)
encoder_state = [state_h, state_c]

# ---- Decoder ----
dec_inputs = keras.layers.Input(shape=(MAX_LEN,), dtype="int64", name="decoder_input")
y = keras.layers.Embedding(VOCAB_TGT, EMB, mask_zero=True)(dec_inputs)
# Condition on encoder state; return_sequences=True because we score every step.
dec_lstm = keras.layers.LSTM(UNITS, return_sequences=True, return_state=True)
dec_seq, _, _ = dec_lstm(y, initial_state=encoder_state)
# Keep outputs as softmax probabilities — greedy/beam decoders below expect
# probabilities. (If you switch to logits + from_logits=True for the loss,
# remember to also change the log-prob computation in beam_search.)
dec_outputs = keras.layers.Dense(VOCAB_TGT, activation="softmax")(dec_seq)

nmt_model = keras.Model([enc_inputs, dec_inputs], dec_outputs)
nmt_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"],
)
nmt_model.summary()


In [ ]:
nmt_model.fit(
    (enc_input_arr, dec_input_arr),
    dec_target_arr,
    batch_size=64,
    epochs=EPOCHS_NMT,
    validation_split=0.1,
)

## 6.6 Inference — greedy decoding

Training used teacher forcing, but at inference the decoder has to feed its own predictions back. This is the loop:

```
state = encoder(source)
decoder_input = [startofseq]
repeat:
    next_token_logits = decoder(decoder_input, state)
    next_token = argmax(next_token_logits[-1])
    decoder_input += next_token
until next_token == endofseq or length limit
```

We'll stick with the full `nmt_model` and just call it repeatedly. It's not the most efficient inference (re-encoding each time), but it's the clearest.

In [ ]:
tgt_vocab = tgt_vec.get_vocabulary()
start_id = tgt_vocab.index("startofseq")
end_id   = tgt_vocab.index("endofseq")

def translate(sentence: str, max_out: int = MAX_LEN) -> str:
    """Greedy decoding: at each step take argmax, feed back as next input."""
    # Keep both inputs as tf.Tensors — Keras 3 refuses mixed tf.Tensor/np.array
    # inputs when you call the model directly.
    enc_in = src_vec(np.array([sentence.lower()]))               # tf.Tensor
    dec_in_arr = np.zeros((1, MAX_LEN), dtype=np.int64)
    dec_in_arr[0, 0] = start_id

    for t in range(1, max_out):
        dec_in = tf.constant(dec_in_arr)
        # Direct call — much faster than .predict() inside a loop because it
        # skips the data-adapter / logging overhead on every step.
        logits = nmt_model([enc_in, dec_in], training=False).numpy()
        next_id = int(np.argmax(logits[0, t - 1]))
        dec_in_arr[0, t] = next_id
        if next_id == end_id:
            break

    words = [tgt_vocab[i] for i in dec_in_arr[0, 1:] if i not in (0, start_id, end_id)]
    return " ".join(words)

for s in ["i like soccer", "where is the hotel?", "it is raining heavily outside today"]:
    print(f"EN: {s}")
    print(f"ES: {translate(s)}\n")


# 7. Beam Search

## 7.1 The Problem with Greedy Decoding

Greedy decoding commits to the single highest-probability token at each step. But the locally-best token is not always part of the globally-best sentence. Example:

```
t=1:  P(I) = 0.6,  P(It) = 0.4
t=2:  after "I" —— best next = "like" (0.3)
      after "It" —— best next = "is" (0.8)
```

Greedy picks "I" at t=1 (higher probability) and ends up with "I like" (0.6 × 0.3 = 0.18).
The path "It is" scores 0.4 × 0.8 = 0.32 — *better overall*, but invisible to greedy.

## 7.2 The idea

Keep the top-**k** partial hypotheses ("beams") at each step. Expand all of them, score by cumulative log-probability, prune back to top-k. At the end, return the best complete hypothesis.

- **k = 1**: equivalent to greedy.
- **k = 5-10**: typical for translation.
- **k → ∞**: exhaustive, exponentially expensive.

## 7.3 Length normalization

Longer sentences have lower joint probability simply because there are more multiplications. Without correction, beam search prefers very short outputs. The fix is to divide log-probability by length (or length^α for some α ∈ [0.6, 1.0]):

$$\text{score}(\hat{y}) = \frac{1}{|\hat{y}|^\alpha} \sum_t \log P(y_t | y_{<t}, x)$$

## 7.4 A sketch implementation

The full production code is intricate; this sketch shows the mechanics.

In [ ]:
def beam_search(sentence: str, beam_width: int = 10,
                max_out: int = MAX_LEN, alpha: float = 0.7):
    """Length-normalized beam search.
    Keeps the top-`beam_width` partial hypotheses at each step.
    Once a beam emits `endofseq` it stops extending but stays in the pool.
    """
    enc_in = src_vec(np.array([sentence.lower()]))  # tf.Tensor

    # Each beam: (tokens_so_far, log_prob, done?)
    beams = [(np.array([start_id] + [0] * (MAX_LEN - 1)), 0.0, False)]

    for t in range(1, max_out):
        if all(b[2] for b in beams):
            break
        candidates = []
        for tokens, lp, done in beams:
            if done:
                candidates.append((tokens, lp, True))
                continue
            # Convert numpy -> tf.Tensor so both inputs to the model are tensors.
            dec_in = tf.constant(tokens.reshape(1, -1))
            # Direct call (not .predict()) — beam search makes many calls per
            # sentence, so the overhead compounds quickly if we use .predict().
            logits = nmt_model([enc_in, dec_in], training=False).numpy()[0, t - 1]
            log_probs = np.log(np.clip(logits, 1e-12, 1.0))  # log-softmax-ish
            top_k = np.argsort(log_probs)[-beam_width:]
            for nxt in top_k:
                new_tokens = tokens.copy()
                new_tokens[t] = int(nxt)
                new_lp = lp + float(log_probs[nxt])
                candidates.append((new_tokens, new_lp, int(nxt) == end_id))

        # Length-normalized score to avoid bias toward shorter sequences.
        def score(beam):
            toks, lp, _ = beam
            length = max(int((toks != 0).sum()), 1)
            return lp / (length ** alpha)

        candidates.sort(key=score, reverse=True)
        beams = candidates[:beam_width]

    best = beams[0][0]
    words = [tgt_vocab[i] for i in best[1:] if i not in (0, start_id, end_id)]
    return " ".join(words)

for s in ["i like soccer", "it is raining heavily outside today"]:
    print(f"EN:          {s}")
    print(f"ES (greedy): {translate(s)}")
    print(f"ES (beam=10): {beam_search(s, beam_width=10)}")
    print()


## 7.5 Beyond beam search

For very large beam widths, pure beam search produces *worse* translations — the model's top candidates become too similar, and you lose diversity. Modern decoders add:

- **Nucleus (top-p) sampling** — sample from the smallest set of tokens whose cumulative probability exceeds p.
- **Temperature** — as in char-RNN.
- **Minimum Bayes risk decoding** — sample many candidates, pick the most consensus.
- **Contrastive search** — penalize tokens that look too similar to recent context.

Note: These are parameters in LLM APIs.